#LLMs for Code Generation

Objective: Convert Python to C++ with speed and accuracy

Whether open or closed use leaderboards to identify candidate LLMs

In [2]:
import os
import io 
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import display, Markdown

In [3]:
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

if not openai_api_key:
    print("OpenAI Key not found")
elif not openai_api_key.startswith("sk-proj"):
    print("Invalid OpenAI Key")
else:
    print("Valid OpenAI Key found and loaded")

if not google_api_key:
    print("Google Key not found")
elif not google_api_key.startswith("AI"):
    print("Invalid Google Key")
else:
    print("Valid Google Key found and loaded")

Valid OpenAI Key found and loaded
Valid Google Key found and loaded


In [4]:
openai = OpenAI()

google_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=google_url, api_key=google_api_key)

In [5]:
GPT_MODEL = "gpt-5-nano"
GEMINI_MODEL = "gemini-2.5-flash-lite"

In [6]:
from system_info import retrieve_system_info
system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '24.6.0',
  'version': 'Darwin Kernel Version 24.6.0: Mon Jan 19 22:01:08 PST 2026; root:xnu-11417.140.69.708.3~1/RELEASE_ARM64_T8112',
  'kernel': '24.6.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin24.6.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M2',
  'cores_logical': 8,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 17.0.0 (clang-1700.6.3.2)',
   'g++': 'Apple clang version 17.0.0 (clang-1700.6.3.2)',
   'clang': 'Apple clang version 17.0.0 (clang-1700.6.3.2)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

In [15]:
message = f"""
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(
    model=GPT_MODEL,
    messages=[{"role":"user", "content": message}]
)

display(Markdown(response.choices[0].message.content))

Short answer
- You likely already have a C++ compiler on macOS. The system info shows Apple clang (in the Xcode Command Line Tools) as the compiler. You don’t need to install anything extra unless you don’t have CLT installed yet.
- If CLT isn’t installed, install it with:
  - xcode-select --install
  - If prompted, follow the on-screen steps (or install Xcode from the App Store and then run xcode-select --install to get CLT).

How to compile and run a single file (simplest way)
- Open Terminal
- Check that a compiler exists:
  - clang++ --version
  - If you get a version, you’re good. If not, install CLT as above.

- Compile (fastest runtime with reasonable optimization):
  - clang++ -std=c++20 -O3 main.cpp -o main
  - Note: -std=c++20 is fine for modern code; you can use -std=c++17 or -std=c++23 if your code requires it.

- Run:
  - ./main

If you prefer to do this from Python (filling in your requested compile_command and run_command)
- Use the following commands:

compile_command = ["clang++", "-std=c++20", "-O3", "main.cpp", "-o", "main"]
run_command = ["./main"]

Example Python snippet (as you showed):
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout

Extra notes
- On macOS, g++ is usually clang++ under the hood; clang++ is perfectly fine to use.
- If you ever install LLVM via Homebrew (brew install llvm) and want to use that clang++, you’d call /usr/local/opt/llvm/bin/clang++ or /opt/homebrew/opt/llvm/bin/clang++ on Apple Silicon, but for most users the system clang++ from Xcode CLT is adequate.
- If your code uses newer C++ features, adjust -std=c++XX accordingly (e.g., -std=c++23).

In [7]:
compile_command = ["clang++", "-std=c++17", "-Ofast", "-mcpu=native", "-flto=thin", "-fvisibility=hidden", "-DNDEBUG", "main.cpp", "-o", "main"]
run_command = ["./main"]

In [8]:
system_prompt = """  
Your task is to convert Python code to high performance C++.capitalizeRespond only with C++ code.
Do not provide any expanation other than opccasional comments.
The C++ response need to produce an identical output in the fastest possible time
"""

def user_prompt_for(python):
    return f"""  
Port this pyhton code to C++ with the fastest possible implementation that produces identical output in the least time.

The system information is {system_info}.

Your response will be written in a file called main.cpp, compiled and then executed. The compilation command is {compile_command}

respond only with C++ Code

Python code to port:

'''python
{python}
'''
""" 

In [9]:
def message_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [10]:
def write_output(cpp):
    with open("main.cpp", 'w', encoding='utf-8') as f:
        f.write(cpp)

In [11]:
def port(client, model, python):
    reasoning_effort = 'high' if 'gpt' in model else None
    response = client.chat.completions.create(
        model=model,
        messages=message_for(python),
        reasoning_effort=reasoning_effort
    )
    result = response.choices[0].message.content
    result = result.replace("'''cpp",'').replace("'''","")
    write_output(result)

In [12]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [13]:
def run_python(code):
    globals={"__builtins__": __builtins__}
    exec(code, globals) #exec allows use to run code that is in string format



In [14]:
run_python(pi)

Result: 3.141592656089
Execution Time: 14.543383 seconds


In [15]:
port(gemini, GEMINI_MODEL, pi)

In [ ]:
# use code below after porting to evaluate performance of the code written
def compile_and_run():
    subprocess.run(compile_command, check=True, text=True, capture_output=True)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)
    print(subprocess.run(run_command, check=True, text=True, capture_output=True).stdout)

port(gemini, GEMINI_MODEL, pi)
compile_and_run()

port(openai, GPT_MODEL, pi)
compile_and_run()

In [ ]:
#replace denominators with model execution times

print(f"""
In our experiments, the performance speedups were:

4th place: Claude Sonnet 4.5: {14.543383 /0.104241:.0f}X speedup 
3rd place: GPT-5: {14.543383 /0.082168:.0f}X speedup
2nd place: Grok 4: {14.543383 /0.018092:.0f}X speedup
1st place: Gemini 2.5 Pro: {114.543383/0.013314:.0f}X speedup
""")
